In [26]:
# %%
import os
import gc
import shutil
import random

import pandas as pd
import numpy as np

import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [27]:
# %%
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [28]:
# %%
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/shohagranasuvo/emotion-sentiment-dataset-csv/Emotion_Sentiment_DataSet.csv


In [6]:
# %%
DATASET_1_PATH = "/kaggle/input/datasets/shohagranasuvo/emotion-sentiment-dataset-csv/Emotion_Sentiment_DataSet.csv"


TEXT_COLUMN = "Text"
LABEL_COLUMN = "Emotion"

In [2]:
# %%
MODEL_NAME = "bert-base-uncased"

MAX_SEQUENCE_LENGTH = 128

EPOCHS = 5

WARMUP_RATIO = 0.1

DROPOUT = 0.1

EXPERIMENTS = [
    {
        "learning_rate": 0.00002,
        "batch_size": 16,
        "weight_decay": 0.01
    },
    {
        "learning_rate": 0.00003,
        "batch_size": 16,
        "weight_decay": 0.01
    },
    {
        "learning_rate": 0.00002,
        "batch_size": 32,
        "weight_decay": 0.01
    },
    {
        "learning_rate": 0.00003,
        "batch_size": 32,
        "weight_decay": 0.01
    },
    {
        "learning_rate": 0.00002,
        "batch_size": 16,
        "weight_decay": 0.1
    },
    {
        "learning_rate": 0.00003,
        "batch_size": 16,
        "weight_decay": 0.1
    },
    {
        "learning_rate": 0.00002,
        "batch_size": 32,
        "weight_decay": 0.1
    },
    {
        "learning_rate": 0.00003,
        "batch_size": 32,
        "weight_decay": 0.1
    }
]

print("Number of BERT configurations:", len(EXPERIMENTS))

Number of BERT configurations: 8


In [4]:
# %%
def prepare_dataset(file_path):

    df = pd.read_csv(file_path)

    print("Original shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nMissing values:")
    print(df.isnull().sum())

    df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)

    print("\nOriginal class distribution:")
    print(df[LABEL_COLUMN].value_counts())

    min_sample = df[LABEL_COLUMN].value_counts().min()

    df_resampled = pd.concat(
        [
            group.sample(
                n=min_sample,
                random_state=73
            )
            for _, group in df.groupby(LABEL_COLUMN)
        ],
        ignore_index=True
    )

    print("\nBalanced class distribution:")
    print(df_resampled[LABEL_COLUMN].value_counts())

    label_encoder = LabelEncoder()

    df_resampled["label"] = label_encoder.fit_transform(
        df_resampled[LABEL_COLUMN]
    )

    train_df, temp_df = train_test_split(
        df_resampled,
        test_size=0.2,
        random_state=73,
        stratify=df_resampled["label"]
    )

    test_df, val_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=73,
        stratify=temp_df["label"]
    )

    print("\nDataset split:")
    print("Train:", train_df.shape)
    print("Validation:", val_df.shape)
    print("Test:", test_df.shape)

    train_dataset = Dataset.from_pandas(
        train_df[[TEXT_COLUMN, "label"]]
    )

    val_dataset = Dataset.from_pandas(
        val_df[[TEXT_COLUMN, "label"]]
    )

    test_dataset = Dataset.from_pandas(
        test_df[[TEXT_COLUMN, "label"]]
    )

    return (
        train_dataset,
        val_dataset,
        test_dataset,
        label_encoder
    )

In [32]:
# %%
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded:", MODEL_NAME)

Tokenizer loaded: bert-base-uncased


In [33]:
# %%
def tokenize_function(examples):

    return tokenizer(
        examples[TEXT_COLUMN],
        padding="max_length",
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH
    )

In [34]:
# %%
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [35]:
# %%
import os
import gc
import torch

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [36]:
# %%
def train_experiment(
    dataset_name,
    experiment_number,
    config,
    train_dataset,
    val_dataset,
    test_dataset,
    label_encoder
):

    num_labels = len(
        label_encoder.classes_
    )

    id2label = {
        i: class_name
        for i, class_name in enumerate(
            label_encoder.classes_
        )
    }

    label2id = {
        class_name: i
        for i, class_name in enumerate(
            label_encoder.classes_
        )
    }

    output_dir = (
        f"/kaggle/working/"
        f"BERT_results/"
        f"{dataset_name}/"
        f"experiment_{experiment_number}"
    )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    print()
    print("=" * 80)
    print(
        f"{dataset_name} - "
        f"Experiment {experiment_number}/8"
    )
    print("=" * 80)

    print(
        "Learning Rate:",
        config["learning_rate"]
    )

    print(
        "Batch Size:",
        config["batch_size"]
    )

    print(
        "Weight Decay:",
        config["weight_decay"]
    )

    print(
        "Output Directory:",
        output_dir
    )

    checkpoint = None

    checkpoints = []

    for directory in os.listdir(output_dir):

        checkpoint_path = os.path.join(
            output_dir,
            directory
        )

        if (
            directory.startswith("checkpoint-")
            and os.path.isdir(checkpoint_path)
        ):

            try:

                step = int(
                    directory.split("-")[1]
                )

                checkpoints.append(
                    (
                        step,
                        checkpoint_path
                    )
                )

            except ValueError:

                continue

    checkpoints.sort(
        key=lambda x: x[0],
        reverse=True
    )

    print()

    if len(checkpoints) == 0:

        print(
            "No checkpoint found."
        )

        print(
            "Training will start from the beginning."
        )

    else:

        print(
            f"Found {len(checkpoints)} "
            f"checkpoint(s)."
        )

        for step, candidate in checkpoints:

            print()
            print(
                f"Checking checkpoint-{step}..."
            )

            required_files = [
                "optimizer.pt",
                "scheduler.pt",
                "trainer_state.json"
            ]

            checkpoint_valid = True

            for filename in required_files:

                file_path = os.path.join(
                    candidate,
                    filename
                )

                if not os.path.exists(
                    file_path
                ):

                    print(
                        f"Missing file: "
                        f"{filename}"
                    )

                    checkpoint_valid = False

                    break

                if os.path.getsize(
                    file_path
                ) == 0:

                    print(
                        f"Empty file: "
                        f"{filename}"
                    )

                    checkpoint_valid = False

                    break

            model_safetensors = os.path.join(
                candidate,
                "model.safetensors"
            )

            model_bin = os.path.join(
                candidate,
                "pytorch_model.bin"
            )

            if not (
                os.path.exists(
                    model_safetensors
                )
                or
                os.path.exists(
                    model_bin
                )
            ):

                print(
                    "Missing model weights."
                )

                checkpoint_valid = False

            rng_file = os.path.join(
                candidate,
                "rng_state.pth"
            )

            if (
                checkpoint_valid
                and os.path.exists(rng_file)
            ):

                if os.path.getsize(
                    rng_file
                ) == 0:

                    print(
                        "Empty file: "
                        "rng_state.pth"
                    )

                    checkpoint_valid = False

            if checkpoint_valid:

                print()
                print(
                    "Valid checkpoint found:"
                )

                print(
                    candidate
                )

                checkpoint = candidate

                break

            else:

                print()
                print(
                    "Invalid checkpoint:"
                )

                print(
                    candidate
                )

    print()

    model = AutoModelForSequenceClassification.from_pretrained(

        MODEL_NAME,

        num_labels=num_labels,

        id2label=id2label,

        label2id=label2id,

        hidden_dropout_prob=DROPOUT,

        attention_probs_dropout_prob=DROPOUT
    )

    training_args = TrainingArguments(

        output_dir=output_dir,

        eval_strategy="steps",

        eval_steps=500,

        save_strategy="steps",

        save_steps=500,

        save_total_limit=2,

        learning_rate=config[
            "learning_rate"
        ],

        per_device_train_batch_size=config[
            "batch_size"
        ],

        per_device_eval_batch_size=config[
            "batch_size"
        ],

        num_train_epochs=EPOCHS,

        weight_decay=config[
            "weight_decay"
        ],

        warmup_ratio=WARMUP_RATIO,

        load_best_model_at_end=True,

        metric_for_best_model="f1",

        greater_is_better=True,

        logging_strategy="steps",

        logging_steps=500,

        report_to="none",

        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=train_dataset,

        eval_dataset=val_dataset,

        compute_metrics=compute_metrics,

        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=2
            )
        ]
    )

    if checkpoint is not None:

        print()
        print("=" * 80)
        print(
            "RESUMING FROM CHECKPOINT"
        )
        print("=" * 80)

        print(
            checkpoint
        )

        print()

        try:

            trainer.train(
                resume_from_checkpoint=checkpoint
            )

        except RuntimeError as error:

            error_message = str(error)

            if (
                "PytorchStreamReader"
                in error_message
                or
                "failed finding central directory"
                in error_message
            ):

                print()
                print("=" * 80)
                print(
                    "CHECKPOINT IS CORRUPTED"
                )
                print("=" * 80)

                print(
                    "The checkpoint could "
                    "not be loaded."
                )

                print(
                    "Deleting corrupted "
                    "checkpoint..."
                )

                shutil.rmtree(
                    checkpoint,
                    ignore_errors=True
                )

                print(
                    "Corrupted checkpoint "
                    "deleted."
                )

                print()
                print(
                    "Starting training "
                    "from the beginning..."
                )

                trainer.train()

            else:

                raise error

    else:

        print()
        print("=" * 80)
        print(
            "STARTING NEW TRAINING"
        )
        print("=" * 80)

        print()

        trainer.train()

    print()
    print("=" * 80)
    print(
        "Evaluating on test dataset..."
    )
    print("=" * 80)

    print()

    test_results = trainer.evaluate(
        test_dataset
    )

    result = {

        "Model": "BERT",

        "Dataset": dataset_name,

        "Experiment": experiment_number,

        "Learning Rate": config[
            "learning_rate"
        ],

        "Batch Size": config[
            "batch_size"
        ],

        "Weight Decay": config[
            "weight_decay"
        ],

        "Acc": test_results[
            "eval_accuracy"
        ],

        "Prec": test_results[
            "eval_precision"
        ],

        "Rec": test_results[
            "eval_recall"
        ],

        "F1": test_results[
            "eval_f1"
        ]
    }

    print()
    print("=" * 80)
    print(
        f"{dataset_name} - "
        f"Experiment {experiment_number} Results"
    )
    print("=" * 80)

    print(
        f"Learning Rate: "
        f"{config['learning_rate']}"
    )

    print(
        f"Batch Size: "
        f"{config['batch_size']}"
    )

    print(
        f"Weight Decay: "
        f"{config['weight_decay']}"
    )

    print()

    print(
        f"Accuracy : "
        f"{result['Acc']:.4f}"
    )

    print(
        f"Precision: "
        f"{result['Prec']:.4f}"
    )

    print(
        f"Recall   : "
        f"{result['Rec']:.4f}"
    )

    print(
        f"F1       : "
        f"{result['F1']:.4f}"
    )

    print("=" * 80)

    del trainer

    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

    return result

In [37]:
# %%
import os

base_dir = "/kaggle/working/BERT_results"

for root, dirs, files in os.walk(base_dir):

    for directory in dirs:

        if directory.startswith("checkpoint-"):

            checkpoint_path = os.path.join(
                root,
                directory
            )

            print()
            print("=" * 70)
            print(checkpoint_path)
            print("=" * 70)

            for file in os.listdir(
                checkpoint_path
            ):

                file_path = os.path.join(
                    checkpoint_path,
                    file
                )

                if os.path.isfile(file_path):

                    size_mb = (
                        os.path.getsize(file_path)
                        / (1024 * 1024)
                    )

                    print(
                        f"{file:<35} "
                        f"{size_mb:.2f} MB"
                    )


/kaggle/working/BERT_results/Dataset 1/experiment_1/checkpoint-2238
config.json                         0.00 MB
trainer_state.json                  0.00 MB
rng_state.pth                       0.01 MB
scheduler.pt                        0.00 MB
optimizer.pt                        407.56 MB
training_args.bin                   0.00 MB
model.safetensors                   417.69 MB
scaler.pt                           0.00 MB


In [38]:
# %%
def run_dataset(
    dataset_name,
    file_path
):

    print()
    print("=" * 80)
    print(dataset_name)
    print("=" * 80)

    (
        train_dataset,
        val_dataset,
        test_dataset,
        label_encoder
    ) = prepare_dataset(file_path)

    train_dataset = train_dataset.map(
        tokenize_function,
        batched=True
    )

    val_dataset = val_dataset.map(
        tokenize_function,
        batched=True
    )

    test_dataset = test_dataset.map(
        tokenize_function,
        batched=True
    )

    remove_columns_train = [
        column
        for column in [
            TEXT_COLUMN,
            "__index_level_0__"
        ]
        if column in train_dataset.column_names
    ]

    remove_columns_val = [
        column
        for column in [
            TEXT_COLUMN,
            "__index_level_0__"
        ]
        if column in val_dataset.column_names
    ]

    remove_columns_test = [
        column
        for column in [
            TEXT_COLUMN,
            "__index_level_0__"
        ]
        if column in test_dataset.column_names
    ]

    train_dataset = train_dataset.remove_columns(
        remove_columns_train
    )

    val_dataset = val_dataset.remove_columns(
        remove_columns_val
    )

    test_dataset = test_dataset.remove_columns(
        remove_columns_test
    )

    dataset_results = []

    for experiment_number, config in enumerate(
        EXPERIMENTS,
        start=1
    ):

        print()
        print("#" * 80)
        print(
            f"{dataset_name} - "
            f"Experiment {experiment_number}/8"
        )
        print("#" * 80)

        print(
            "Learning Rate:",
            config["learning_rate"]
        )

        print(
            "Batch Size:",
            config["batch_size"]
        )

        print(
            "Weight Decay:",
            config["weight_decay"]
        )

        result = train_experiment(
            dataset_name,
            experiment_number,
            config,
            train_dataset,
            val_dataset,
            test_dataset,
            label_encoder
        )

        dataset_results.append(result)

        print()
        print("Results")
        print(
            f"Accuracy : {result['Acc']:.4f}"
        )
        print(
            f"Precision: {result['Prec']:.4f}"
        )
        print(
            f"Recall   : {result['Rec']:.4f}"
        )
        print(
            f"F1       : {result['F1']:.4f}"
        )

        print()

    return dataset_results

In [39]:
# %%
dataset_1_results = run_dataset(
    "Dataset 1",
    DATASET_1_PATH
)


Dataset 1
Original shape: (160000, 3)

Columns:
['Unnamed: 0', 'Text', 'Emotion']

Missing values:
Unnamed: 0    0
Text          8
Emotion       0
dtype: int64

Original class distribution:
Emotion
love          39553
happiness     27175
sadness       17481
Normal        16351
hate          15267
anger         12336
Depression    10333
fun           10075
surprise       6954
worry          4475
Name: count, dtype: int64

Balanced class distribution:
Emotion
Depression    4475
Normal        4475
anger         4475
fun           4475
happiness     4475
hate          4475
love          4475
sadness       4475
surprise      4475
worry         4475
Name: count, dtype: int64

Dataset split:
Train: (35800, 4)
Validation: (4475, 4)
Test: (4475, 4)


Map:   0%|          | 0/35800 [00:00<?, ? examples/s]

Map:   0%|          | 0/4475 [00:00<?, ? examples/s]

Map:   0%|          | 0/4475 [00:00<?, ? examples/s]


################################################################################
Dataset 1 - Experiment 1/8
################################################################################
Learning Rate: 2e-05
Batch Size: 16
Weight Decay: 0.01

Dataset 1 - Experiment 1/8
Learning Rate: 2e-05
Batch Size: 16
Weight Decay: 0.01
Output Directory: /kaggle/working/BERT_results/Dataset 1/experiment_1

Found 1 checkpoint(s).

Checking checkpoint-2238...

Valid checkpoint found:
/kaggle/working/BERT_results/Dataset 1/experiment_1/checkpoint-2238



...

Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
500,1.785390,0.132508,0.983687,0.984255,0.983677,0.983754
1000,0.074953,0.060580,0.992849,0.992891,0.992850,0.992846


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:668] . unexpected pos 785308800 vs 785308688